# Preprocess (all-genre, streaming)

Runs on **Kaggle CPU** — no GPU needed. Preprocessing now runs **all-genre on ~16 GB RAM via the streaming preprocessor** (a few disk passes, holds only counts in RAM), and still works on Kaggle CPU.
Outputs `sample.parquet` and `catalog.parquet` to `/kaggle/working/`; download those to your laptop's `artifacts/`, then embed locally with `03_embed_local.ipynb`. See `docs/kaggle-setup.md`.

In [ ]:
# If running from a fresh clone:
# !git clone <your-repo-url> && %cd book_recsys && pip install -e .
!pip install -q pyarrow

In [ ]:
import glob
import os
import pandas as pd
from book_recsys.data.streaming import streaming_kcore_sample
from book_recsys.data.books import stream_books_json
from book_recsys.data.schema import BOOK

In [ ]:
INTERACTION_FILES = sorted(glob.glob("data/interactions_*.json.gz"))  # all downloaded genres
BOOKS = "data/books.json.gz"
OUT = "artifacts"
os.makedirs(OUT, exist_ok=True)
print(f"{len(INTERACTION_FILES)} genre files:", INTERACTION_FILES)

In [ ]:
# Streaming all-genre k-core + 50k-user sample -> sample.parquet (memory-bounded).
summary = streaming_kcore_sample(INTERACTION_FILES, min_user=20, min_book=10,
                                 n_users=50_000, seed=42, out_path=f"{OUT}/sample.parquet")
summary

In [ ]:
# Catalog = book metadata for the books that survived sampling, with author names attached.
from book_recsys.data.authors import load_author_names, attach_author_names
keep = set(pd.read_parquet(f'{OUT}/sample.parquet')[BOOK].unique())
books = pd.concat(stream_books_json(BOOKS), ignore_index=True)
catalog = books[books['book_id'].isin(keep)].reset_index(drop=True)
# author_id + work_id come from stream_books_json; resolve author_id -> name (docs + UI labels)
authors_file = next((g for n in ['goodreads_book_authors.json.gz', 'goodreads_book_authors.json',
                                 'book_authors.json.gz', 'book_authors.json']
                     for g in glob.glob(f'data/{n}') + glob.glob(f'/kaggle/input/**/{n}', recursive=True)),
                    None)
if authors_file:
    catalog = attach_author_names(catalog, load_author_names(authors_file))
    print('attached author names from', authors_file)
else:
    print('WARNING: author-names file not found -> catalog has author_id but NO `author` name column')
catalog.to_parquet(f'{OUT}/catalog.parquet')
print(catalog.shape, '| columns:', list(catalog.columns))

**Next:** Use **Save Version** to persist `/kaggle/working`, then download `sample.parquet` and `catalog.parquet` into the repo's `artifacts/` folder. Run `03_embed_local.ipynb` on your Mac to produce `embeddings.npy` and `faiss.idx`.

In [ ]:
catalog